# Project 2: Semantic Search

## Model exploration

###  Importing steam game description data

Databases are downloaded from https://www.kaggle.com/datasets/nikdavis/steam-store-games

In [3]:
import pandas as pd
df = pd.read_csv('data/steam_description_data.csv')

### Combining the two descriptions into one

In [4]:
df['combined_description'] = df['about_the_game'] + " " + df['detailed_description']
df.head(5)


,steam_appid,detailed_description,about_the_game,short_description,combined_description
0,10,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...
1,20,One of the most popular online action games of...,One of the most popular online action games of...,One of the most popular online action games of...,One of the most popular online action games of...
2,30,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...
3,40,Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...
4,50,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...


In [5]:
descriptions = df['combined_description'].tolist()

for x in range(0,5):
    print(descriptions[x])

Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role. Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.
One of the most popular online action games of all time, Team Fortress Classic features over nine character classes -- from Medic to Spy to Demolition Man -- enlisted in a unique style of online team warfare. Each character class possesses unique weapons, items, and abilities, as teams compete online in a variety of game play modes. One of the most popular online ac

### Testing the transformer on the database

In [6]:
import torch
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

Device: mps


In [7]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

corpus_embedding = embedder.encode(descriptions, 
                                   convert_to_tensor=True, 
                                   device=device)

queries = ["What is the chillest farming simulator ever?",
           "Horror real-time strategy game",
           "What is the most realistic first-person shooter game?"]

query_embedding = embedder.encode(queries, 
                                  convert_to_tensor=True, 
                                  device=device)

result = util.semantic_search(query_embedding, 
                              corpus_embedding, 
                              top_k=5)

print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'corpus_id': 1497, 'score': 0.677585780620575}, {'corpus_id': 8257, 'score': 0.6693726778030396}, {'corpus_id': 3583, 'score': 0.6475917100906372}, {'corpus_id': 20004, 'score': 0.6472078561782837}, {'corpus_id': 1046, 'score': 0.6316966414451599}], [{'corpus_id': 12235, 'score': 0.6654729843139648}, {'corpus_id': 25178, 'score': 0.642960786819458}, {'corpus_id': 23742, 'score': 0.6351367831230164}, {'corpus_id': 21108, 'score': 0.617480456829071}, {'corpus_id': 8543, 'score': 0.6030283570289612}], [{'corpus_id': 8438, 'score': 0.6228830814361572}, {'corpus_id': 5693, 'score': 0.602461040019989}, {'corpus_id': 18180, 'score': 0.5937350392341614}, {'corpus_id': 23715, 'score': 0.5910769104957581}, {'corpus_id': 23057, 'score': 0.5879915952682495}]]


### Save the corpus embbeding variable into numpy

In [8]:
import numpy as np


np.save('embeddings.npy', corpus_embedding.cpu().numpy())

## Semantic search implementation

### Load model and corpus embbeding

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer, util


device = "mps" if torch.backends.mps.is_available() else "cpu"

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
corpus_embeddings = np.load('embeddings.npy')
corpus_embeddings = torch.from_numpy(corpus_embeddings).to(device)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Load data again for formatting purpose

In [ ]:

df_descriptions = pd.read_csv('data/steam_description_data.csv')
df_games = pd.read_csv('data/steam.csv')
game_lookup = dict(zip(df_games['appid'], df_games['name']))

### Semantic search function

In [ ]:
def semantic_search(query, top_k = 5):
    """
    Search for games using query
    
    Args:
        query (str): Search query
        top_k (int): Number of results (K) to return
    
    Returns:
        list: Top K results with game names and similarity scores
    """

    query_embedding = embedder.encode(query,
                                      convert_to_tensor=True,
                                      device=device)

    result = util.semantic_search(query_embedding, 
                              corpus_embedding, 
                              top_k=top_k)
    
    formatted_res = []
    for rank, result in enumerate(result[0], 1):

        corpus_id = result['corpus_id'] 
        score = result['score']
        
        steam_appid = df_descriptions.iloc[corpus_id]['steam_appid']
        
        game_name = game_lookup.get(steam_appid, "Unknown Game")
        
        formatted_res.append({
            'rank': rank,
            'game': game_name,
            'score': score
        })
    
    return formatted_res
    



    


### Test the function

In [12]:
results = semantic_search("sci-fi real-time strategy", top_k=5)
for r in results:
    print(f"{r['rank']}. {r['game']} (Score: {r['score']:.3f})")

1. Ways of History (Score: 0.574)
2. 冒险之路(Adventure Road) (Score: 0.563)
3. Lootfest Wars (Score: 0.541)
4. Wayward Terran Frontier: Zero Falls (Score: 0.528)
5. Battle Of Worldviews (Score: 0.528)


### Test the function using multiple queries

In [13]:
test_queries = [
    "action rpg",
    "visual novel slice of life",
    "coop survival",
    "escape room",
    "medieval open world",
    "cozy farming simulator",
    "battle royale",
    "rougelike"
]

# Run searches
for query in test_queries:
    print(f"Query: {query}")
    results = semantic_search(query, top_k=3)
    for r in results:
        print(f"{r['rank']}. {r['game']} (Score: {r['score']:.3f})")
    
    print("\n")

Query: action rpg
1. Scrolls of the Lord (Score: 0.593)
2. Brave (Score: 0.589)
3. Fearless Fantasy (Score: 0.588)


Query: visual novel slice of life
1. The Story of My Life (Score: 0.546)
2. 神明的一天世界(God's One Day World) (Score: 0.540)
3. Return of Red Riding Hood Enhanced Edition (Score: 0.525)


Query: coop survival
1. Project Winter (Score: 0.521)
2. How to Survive 2 (Score: 0.515)
3. TRAPPED (Score: 0.504)


Query: escape room
1. CRIMSON ROOM® DECADE (Score: 0.679)
2. Escape (Score: 0.637)
3. Mystery House -fivestones- (Score: 0.624)


Query: medieval open world
1. Regions Of Ruin (Score: 0.578)
2. The Guild 3 (Score: 0.526)
3. Life is Feudal: Your Own (Score: 0.521)


Query: cozy farming simulator
1. Farming Simulator 17 (Score: 0.699)
2. Agricultural Simulator 2013 - Steam Edition (Score: 0.696)
3. Farming Simulator 2011 (Score: 0.689)


Query: battle royale
1. Pixel Battle Royale (Score: 0.674)
2. Super Animal Royale (Score: 0.646)
3. Mortal Royale (Score: 0.628)


Query: rouge